# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates exploration, processing, and visualization of the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
Mass Spectrometry protein analysis from human mast cells, provided via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from this Croissant dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL and load the schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(croissant_url)

# Access and print main metadata. Do not use dict subscripting for metadata
meta = dataset.metadata
print(f"Name: {meta.name}\nVersion: {meta.version}")
print(f"Description: {meta.description}")
if hasattr(meta, 'keywords'):
    print(f"Keywords: {meta.keywords}")
if hasattr(meta, 'datePublished'):
    print(f"Date published: {meta.datePublished}")

## 2. Data Overview
Review available record sets, their IDs (using `@id`), and the fields within each set. All references are by `@id`.

In [ ]:
# List all record sets and their fields using their @id
record_sets = dataset.record_sets()
print(f"Number of record sets: {len(record_sets)}\n")
all_record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}")
    all_record_set_ids.append(rs.id)
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (id: {f.id}, type: {getattr(f, 'data_type', 'unknown')})")
    print()

## 3. Data Extraction
Load data for a record set into a DataFrame. Below, you may use the main protein table (commonly the primary data in mass spec datasets). You can select any record set by its `@id`.

In [ ]:
# Choose a main record set for extraction, such as the first one (adjust as needed)
protein_record_set_id = all_record_set_ids[0]
print(f"Extracting from RecordSet id: {protein_record_set_id}")

records = list(dataset.records(record_set=protein_record_set_id))
df = pd.DataFrame(records)
print(f"Extracted columns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
We identify numeric fields suitable for statistics and visualization. All field references are by their `@id`. Example EDA includes filtering, normalization, and grouping.

In [ ]:
# List all numeric fields (columns of type Integer/Float)
numeric_field_ids = []
rs = [r for r in dataset.record_sets() if r.id == protein_record_set_id][0]
for f in rs.fields:
    dt = getattr(f, "data_type", None)
    if dt in ("Integer", "Float", "Number"):
        numeric_field_ids.append(f.id)
print(f"Numeric @id fields: {numeric_field_ids}")

# Pick a numeric field for demonstration
numeric_field_id = numeric_field_ids[0] if numeric_field_ids else None
if not numeric_field_id:
    print("No numeric fields found.")
else:
    # Filter entries where the field > a threshold
    threshold = df[numeric_field_id].quantile(0.75)  # upper quartile as example
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    mean = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    std = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - mean) / std
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (e.g., accession/description)
    # Find a candidate group-by field
    group_field = None
    for f in rs.fields:
        # Use first non-numeric, text-like field
        if getattr(f, "data_type", "") in ("Text", "String"):
            group_field = f.id
            break
    print(f"\nGrouping field: {group_field}")
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped mean by {group_field} (showing first 5 entries):")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field (`@id: {}`), highlighting normalized values and optionally grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If normalized column exists, plot it
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[norm_col].dropna(), bins=30, color='orange', kde=True)
        plt.title(f"Normalized {numeric_field_id} for filtered entries")
        plt.xlabel(norm_col)
        plt.ylabel("Count")
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 mass spectrometry dataset using `mlcroissant`, referencing all dataset elements by `@id` as per the Croissant specification. We demonstrated loading, field enumeration, processing, and visualization workflows suitable for further analysis such as biomarker discovery or comparative protein studies.